<a href="https://colab.research.google.com/github/edgaralbasanz/ealba-xatbot/blob/main/XatBot2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
from google.colab import userdata
import google.generativeai as genai
import requests
from bs4 import BeautifulSoup
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok
import re

# =================== SECRETS ===================
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
genai.configure(api_key=GEMINI_API_KEY)

model = genai.GenerativeModel(
    'gemini-2.5-flash',
    generation_config={
        "temperature": 0.7,
        "max_output_tokens": 800,
    }
)

app = Flask(__name__)
CORS(app)

WORDPRESS_URL = "https://ealba.inscastellbisbal.net"  # ← La teva URL

def clean_text(text):
    text = re.sub(r'\*+', '', text)
    text = re.sub(r'\n\s*\n', '\n\n', text)
    return text.strip()

def scrape_page(url, headers):
    """Scraping d'una pàgina individual"""
    try:
        r = requests.get(url, headers=headers, timeout=10)
        if r.status_code != 200:
            return ""
        soup = BeautifulSoup(r.text, 'html.parser')

        # Elimina menús, capçaleres i peus de pàgina (soroll)
        for tag in soup.find_all(['nav', 'header', 'footer', 'script', 'style']):
            tag.decompose()

        texts = [
            tag.get_text(separator=' ', strip=True)
            for tag in soup.find_all(['h1', 'h2', 'h3', 'p', 'li', 'strong'])
            if len(tag.get_text(strip=True)) > 20
        ]
        return " ".join(texts[:30])
    except:
        return ""

def get_all_pages_content():
    """Recull contingut de la pàgina principal + totes les pàgines internes"""
    headers = {'User-Agent': 'Mozilla/5.0'}
    all_content = []

    # 1. Pàgina principal
    main_content = scrape_page(WORDPRESS_URL, headers)
    if main_content:
        all_content.append(f"[Pàgina principal]\n{main_content}")

    # 2. Descobrir pàgines internes des del menú
    try:
        r = requests.get(WORDPRESS_URL, headers=headers, timeout=10)
        soup = BeautifulSoup(r.text, 'html.parser')

        # Busca tots els enllaços interns del menú de navegació
        internal_links = set()
        for a in soup.find_all('a', href=True):
            href = a['href']
            if (
                href.startswith(WORDPRESS_URL) and
                href != WORDPRESS_URL and
                not any(x in href for x in ['#', 'wp-', '.jpg', '.png', '.pdf', '?'])
            ):
                internal_links.add(href.rstrip('/'))

        # 3. Scraping de cada pàgina interna (màx. 8 pàgines)
        for url in list(internal_links)[:8]:
            content = scrape_page(url, headers)
            if content:
                # Nom de la pàgina a partir de la URL
                page_name = url.replace(WORDPRESS_URL, '').strip('/')
                all_content.append(f"[Pàgina: /{page_name}]\n{content}")

    except Exception as e:
        print(f"Error descobrint pàgines: {e}")

    return "\n\n---\n\n".join(all_content)


@app.route('/')
def home():
    return "✅ XatBot actiu!"

@app.route('/chat', methods=['POST'])
def chat():
    try:
        user_message = request.json.get('message', '') if request.json else ''

        # Recull contingut de totes les pàgines
        context = get_all_pages_content()

        if not context:
            return jsonify({"reply": "No he pogut llegir la web. Torna-ho a provar."})

        prompt = f"""Ets un assistent professional, amable i directe de Edgar Alba.
Respon sempre en català, de forma natural i sense usar Markdown (** o *).
No facis llistes amb asteriscs. Escriu de forma fluida i conversacional.
Usa únicament la informació proporcionada per respondre. Si no saps la resposta, dis-ho honestament.

Informació de la web:
{context}

Pregunta de l'usuari: {user_message}"""

        response = model.generate_content(prompt)
        clean_reply = clean_text(response.text)

        return jsonify({"reply": clean_reply})

    except Exception as e:
        error_str = str(e).lower()
        if "429" in error_str or "quota" in error_str:
            return jsonify({"reply": "⏳ Estic amb molta demanda ara mateix. Torna-ho a provar d'aquí a 30-60 segons."})
        else:
            print(f"Error: {e}")
            return jsonify({"reply": "Ho sento, hi ha hagut un error. Torna-ho a provar."})


# ===================== INICIAR =====================
public_url = ngrok.connect(5000)
print("\n" + "="*60)
print("✅ XATBOT MILLORAT EN FUNCIONAMENT")
print(f"🔗 URL: {public_url}")
print("="*60)

app.run(port=5000)


✅ XATBOT MILLORAT EN FUNCIONAMENT
🔗 URL: NgrokTunnel: "https://nonoriginally-basidial-fidelia.ngrok-free.dev" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
